In [1]:
from synthetic_workloads import *

IndentationError: expected an indented block after 'with' statement on line 284 (synthetic_workloads.py, line 286)

In [ ]:
results_path = './activations/'
filename_qwen = results_path + 'activations_qwen_samples15000_20260814083534.pkl'
filename_example = results_path + 'activations_mistral_example.pkl'
save_path = './workloads/'

In [ ]:
# Read activation records
model = 'qwen'
results = get_results(filename_qwen)

In [ ]:
# Model parameters
n_experts = results[0]['probs'].shape[1]
n_layers = results[0]['probs'].shape[2]
n_samples = len(results)
k = results[0]['active_experts'].shape[1]

In [ ]:
# Get the "naturally occuring" CV in the data
qs = get_qs(results, weighted_by_token_count=True)
qs_mean = qs.mean(dim=0)
cv_nat = qs_mean.std(dim=0) / qs_mean.mean(dim=0) # (n_layers)

In [ ]:
# Check assumptions on target
rand = qs[torch.randint(len(qs), (64,))].sum(0)
assert (cvs_of(rand, 0) - cv_nat).abs().max() < 0.05 * cv_nat.max(), "pipeline broken"

In [ ]:
# Now search for workloads which fit a range of CVs
# We sweep a parameter alpha which scales cv_nat
target_alphas = [0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 2.0, 2.8, 3.5, 4.5, 6.0]
target_cvs_list = [cv_nat * a for a in target_alphas]
target_ls = [64, 128]
workloads_cvs_repeats0 = workload_sweep_cvs(results, target_cvs_list, target_ls, max_repeats=0, verbose=True)
plot_workload_sweep_cvs(workloads_cvs_repeats0, target_alphas, cv_nat, f'Synthetic Workload Generation, Max Allowed Prompt Repeats=0')

In [ ]:
# Try allowing repeats
max_repeats = 1024
workloads_cvs_repeatsmax = workload_sweep_cvs(results, target_cvs_list, target_ls, max_repeats=max_repeats, verbose=True)
plot_workload_sweep_cvs(workloads_cvs_repeatsmax, target_alphas, cv_nat, f'Synthetic Workload Generation, Max Allowed Prompt Repeats=1024')

In [ ]:
# Try allowing more repeats
#max_repeats = 2
#workloads_cvs_repeats2 = workload_sweep_cvs(results, target_cvs, target_ls, max_repeats=max_repeats, verbose=True)
#plot_workload_sweep_cvs(workloads_cvs_repeats2, f'Synthetic Workload Generation, Max Allowed Prompt Repeats={max_repeats}')

In [ ]:
# Try the same but using PMR as a target rather than CV
#target_pmrs = [0.2, 0.4, 0.6, 0.8]
#workloads_pmrs_repeats0 = workload_sweep_pmrs(results, target_pmrs, target_ls, max_repeats=0, verbose=True)
#plot_workload_sweep_pmrs(workloads_pmrs_repeats0, f'Synthetic Workload Generation, Max Allowed Prompt Repeats=0')

In [ ]:
# Try allowing repeats
#max_repeats = 1024
#workloads_pmrs_repeats1 = workload_sweep_pmrs(results, target_cvs, target_ls, max_repeats=max_repeats, verbose=True)
#plot_workload_sweep_pmrs(workloads_pmrs_repeats1, f'Synthetic Workload Generation, Max Allowed Prompt Repeats={max_repeats}')

In [ ]:
# Try allowing more repeats
#max_repeats = 2
#workloads_pmrs_repeats1 = workload_sweep_cvs(results, target_cvs, target_ls, max_repeats=max_repeats, verbose=True)
#plot_workload_sweep_pmrs(workloads_pmrs_repeats1, f'Synthetic Workload Generation, Max Allowed Prompt Repeats={max_repeats}')

In [ ]:
# Compare to subject workloads
subjects = set([r['subject'] for r in results])
subject_results = {s:[r for r in results if r['subject']==s] for s in subjects}
subject_results = {s:subject_results[s] for s in subject_results if len(subject_results[s])>=max(target_ls)}


In [ ]:
workload_id = f'workloads_{model}'
save_file = save_path + workload_id + '.pkl'
# Save workloads to file
#save_workloads(workloads, save_file)